# BEST SDV SYNTHESIZERS SYNTHETIC DATA EVALUATION

## LOAD SYNTH DATA & ORIGINAL FILE

### Unrar synthesizers 

In [ ]:
import os
import pandas as pd
import patoolib

pd.set_option("display.max_column",None)


# set path
results_folder = "../results"

# unzip best synth data .rar file
patoolib.extract_archive(os.path.join(results_folder,"best_synth_data.rar"), outdir=results_folder)

### Load synthetic data

In [ ]:
# set synth data path
file1 = os.path.join(results_folder,"GaussianCopula_best_synth_data.parquet")
file2 = os.path.join(results_folder,"CTGAN_best_synth_data.parquet")

# Load best synth data
copula = pd.read_parquet(file1,engine="pyarrow")
ctgan = pd.read_parquet(file2,engine="pyarrow")

### Load original data

In [ ]:
# set original data path
orig = os.path.join(results_folder,"preprocessed_file.parquet")

# load original data
diabetes =  pd.read_parquet(orig, engine="pyarrow")

## EVALUATE COLUMNS SHAPE

### Dimensions

In [ ]:
print(f"Real dimension: {diabetes.shape}")
print(f"Copula Synth dimension: {copula.shape}")
print(f"CTGAN Synth dimension: {ctgan.shape}")

### Not Null values

In [ ]:
# Get information from 3 datasets
real_data_info = pd.DataFrame({
    'Column': diabetes.columns,
    'Real Count':diabetes.notnull().sum()
    })

copula_synth_info = pd.DataFrame({
    'Column': copula.columns,
    'GaussianCopula Count':copula.notnull().sum()
})

ctgan_synth_info = pd.DataFrame({
    'Column': ctgan.columns,
    'CTGAN Count':ctgan.notnull().sum()
})


# Merge the  DataFrames on the 'Column' name
comparison = pd.merge(real_data_info, copula_synth_info, on='Column', how='outer')
comparison = pd.merge(comparison, ctgan_synth_info, on='Column', how='outer')

# Print comparison table
print("Comparison of Real and Synthetic Data:")
print(comparison)

### Check categorical values

#### Same categoricals 

In [ ]:
# get categorical column names
orig_cat_cols = diabetes.select_dtypes('object').columns.tolist()
copula_cat_cols = copula.select_dtypes('object').columns.tolist()
ctgan_cat_cols = ctgan.select_dtypes('object').columns.tolist()

# evaluate the result
print(f"Same categorical columns: {orig_cat_cols == copula_cat_cols == ctgan_cat_cols}")

#### Categoricals distribution

In [ ]:
for col in orig_cat_cols:
    # Get information from 3 datasets
    real_data_info = diabetes[col].value_counts(dropna=True).reset_index()
    real_data_info.columns = ['Category', 'Real']

    copula_synth_info = copula[col].value_counts(dropna=True).reset_index()
    copula_synth_info.columns = ['Category', 'GaussianCopula']

    ctgan_synth_info = ctgan[col].value_counts(dropna=True).reset_index()
    ctgan_synth_info.columns = ['Category', 'CTGAN']

    # Merge the DataFrames on the 'Value' name
    comparison = pd.merge(real_data_info, copula_synth_info, on='Category', how='outer')
    comparison = pd.merge(comparison, ctgan_synth_info, on='Category', how='outer')

    # Print comparison table
    print(f"\nComparison of Real and Synthetic Data for column '{col}':")
    print(comparison)

### Check numerical values

#### Same numerial columns 

In [ ]:
# get numerical column names
orig_cols = diabetes.select_dtypes('int64').columns.tolist()
copula_cols = copula.select_dtypes('int64').columns.tolist()
ctgan_cols = ctgan.select_dtypes('int64').columns.tolist()

# evaluate same quantity
print(f"Same numerical columns: {len(orig_cols) == len(copula_cols) == len(ctgan_cols)}")

#### Numericals distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# plot their distribution
for col in orig_cols:
    # Plot histogram
    fig, ax = plt.subplots(1,2,figsize=(10, 5))
    
    # Get unique values count for binning (useful for integer columns)
    unique_values = diabetes[col].nunique()
    
    # Set the number of bins based on unique values or a minimum threshold for better visualization
    if unique_values < 30:
        bins = unique_values  # Use number of unique values if less than 30
    else:
        bins = 30  # Default to 30 bins if more than 30 unique values
    
    # Set xticks based on min and max values in the column
    col_min, col_max = diabetes[col].min(), diabetes[col].max()
    
    # Adjust step size for xticks dynamically (if range is small, step=1, else larger step)
    if col_max - col_min < 30:
        step_size = 1
    else:
        step_size = (col_max - col_min) // 10  # Step size as a fraction of the range
    
    
    # orig vs copula
    ax[0].hist(diabetes[col], bins= bins, alpha = 0.2, label = "orig")
    ax[0].hist(copula[col], bins= bins, alpha = 0.2, label = "copula")
    ax[0].set_xticks(np.arange(col_min, col_max + step_size, step_size))
    
    # Adjust & show the plot
    ax[0].set_title(f'Distribution of {col}')
    ax[0].set_xlabel(col)
    ax[0].set_ylabel('Frequency')
    ax[0].legend()
    
    # orig vs ctgan
    ax[1].set_xticks(np.arange(col_min, col_max + step_size, step_size))    
    ax[1].hist(diabetes[col], bins= bins, alpha = 0.2, label = "orig")
    ax[1].hist(ctgan[col], bins= bins, alpha = 0.2, label = "ctgan")
    
    # Adjust & show the plot
    ax[1].set_title(f'Distribution of {col}')
    ax[1].set_xlabel(col)
    ax[1].set_ylabel('Frequency')
    ax[1].legend()
    
    plt.tight_layout()
    plt.show()

## EVALUATE COLUMNS PAIR

#### Categoricals 

#### Numericals 

## CONCLUSIONS